In [ ]:
"""
Scraping Data Harga Bahan Pokok (Makanan) - Kabupaten Jember
Periode : 2021-01-01 s/d 2025-12-31 (per hari)
Sumber  : https://siskaperbapo.jatimprov.go.id/harga/tabel
Metode  : Selenium headless Chrome (situs butuh JavaScript)

Output  : pangan_makanan_jember_2021_2025.csv
Kolom   : tanggal, komoditas, satuan, harga_lama, harga_sekarang, selisih, persen

Install:
    pip install selenium webdriver-manager

Jalankan:
    python scraping_pangan_jember.py

Tips:
    - Jika terhenti, jalankan lagi → otomatis lanjut dari tanggal terakhir (progress.txt)
    - Estimasi waktu: ~2-3 jam untuk 5 tahun data
"""

import csv
import os
import sys
import time
from datetime import date, timedelta
from bs4 import BeautifulSoup

# ── Konfigurasi ───────────────────────────────────────────────────────────────
TABLE_URL     = "https://siskaperbapo.jatimprov.go.id/harga/tabel"
KABUPATEN     = "Jember"
TAHUN_MULAI   = 2021
TAHUN_AKHIR   = 2025
OUTPUT_FILE   = f"pangan_makanan_jember_{TAHUN_MULAI}_{TAHUN_AKHIR}.csv"
PROGRESS_FILE = "progress.txt"
DELAY_DETIK   = 2.0   # jeda antar tanggal agar tidak di-block
# ─────────────────────────────────────────────────────────────────────────────

KATEGORI_MAKANAN = {
    "BERAS", "GULA", "MINYAK GORENG", "DAGING", "TELUR AYAM",
    "IKAN", "BAWANG", "CABAI", "KEDELAI", "TEPUNG TERIGU",
    "JAGUNG", "GARAM", "KACANG TANAH", "SUSU", "SAYURAN",
    "BUAH", "TAHU", "TEMPE", "MINYAK MAKAN",
    "DAGING AYAM", "DAGING SAPI", "IKAN SEGAR", "IKAN ASIN", "BUMBU",
}

KOLOM = ["tanggal", "komoditas", "satuan",
         "harga_lama", "harga_sekarang", "selisih", "persen"]


# =============================================================================
# PARSER TABEL
# =============================================================================

def parse_tabel(soup: BeautifulSoup, tanggal_str: str) -> list[dict]:
    hasil = []
    kategori_aktif = None

    for tbl in soup.find_all("table"):
        for row in tbl.find_all("tr"):
            cols = [td.get_text(strip=True) for td in row.find_all("td")]
            if len(cols) < 2:
                continue

            nama       = cols[1]
            nama_upper = nama.strip().upper()

            # Baris INDUK
            cocok = next((k for k in KATEGORI_MAKANAN
                          if nama_upper == k or nama_upper.startswith(k)), None)
            if cocok:
                kategori_aktif = nama_upper
                hasil.append({
                    "tanggal": tanggal_str, "komoditas": nama_upper,
                    "satuan": "", "harga_lama": "0", "harga_sekarang": "0",
                    "selisih": "", "persen": "",
                })
                continue

            # Baris ITEM
            if nama.startswith("-") and kategori_aktif:
                hasil.append({
                    "tanggal":        tanggal_str,
                    "komoditas":      nama.strip(),
                    "satuan":         cols[2] if len(cols) > 2 else "",
                    "harga_lama":     _angka(cols[3] if len(cols) > 3 else ""),
                    "harga_sekarang": _angka(cols[4] if len(cols) > 4 else ""),
                    "selisih":        _angka(cols[5] if len(cols) > 5 else ""),
                    "persen":         cols[6] if len(cols) > 6 else "",
                })

    return hasil


def _angka(s: str) -> str:
    return s.replace(".", "").strip()


# =============================================================================
# SELENIUM
# =============================================================================

def init_driver():
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.chrome.service import Service
    from webdriver_manager.chrome import ChromeDriverManager

    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opts,
    )
    print("[OK]  Chrome headless siap.\n")
    return driver


def setup_filter(driver, wait):
    """
    Buka halaman, pilih Kabupaten Jember di dropdown, klik cari.
    Dipanggil sekali di awal — setelah itu cukup ganti tanggal.
    """
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import Select

    driver.get(TABLE_URL)
    time.sleep(3)

    # Pilih Kabupaten Jember
    selects = driver.find_elements(By.TAG_NAME, "select")
    for sel_el in selects:
        attr = (
            (sel_el.get_attribute("name") or "") + " " +
            (sel_el.get_attribute("id")   or "")
        ).lower()
        if any(k in attr for k in ["kabupaten", "kab", "wilayah", "daerah"]):
            try:
                s = Select(sel_el)
                for opt in s.options:
                    if KABUPATEN.lower() in opt.text.lower():
                        s.select_by_visible_text(opt.text)
                        print(f"[OK]  Dropdown kabupaten → '{opt.text}'")
                        break
            except Exception as e:
                print(f"[WARN] Dropdown: {e}")

    _klik_cari(driver)
    time.sleep(2)


def fetch_tanggal(driver, wait, tanggal_str: str) -> list[dict]:
    """
    Ganti input tanggal → klik cari → parse tabel.
    """
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import Select

    # Cari input/select tanggal
    _isi_tanggal(driver, tanggal_str)
    _klik_cari(driver)
    time.sleep(2)

    soup = BeautifulSoup(driver.page_source, "html.parser")
    return parse_tabel(soup, tanggal_str)


def _isi_tanggal(driver, tanggal_str: str):
    """
    Coba isi field tanggal — bisa berupa <input type=date>, <input type=text>,
    atau <select> bulan/tahun terpisah.
    """
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import Select
    from selenium.webdriver.common.keys import Keys

    tgl = date.fromisoformat(tanggal_str)

    # ── Coba input[type=date] atau input[name*=tanggal] ──
    inputs = driver.find_elements(
        By.XPATH,
        "//input[@type='date' or contains(@name,'tanggal') or "
        "contains(@id,'tanggal') or contains(@name,'tgl') or contains(@id,'tgl')]"
    )
    for inp in inputs:
        try:
            inp.clear()
            # Format bisa YYYY-MM-DD atau DD-MM-YYYY tergantung situs
            inp.send_keys(tanggal_str)
            return
        except Exception:
            pass

    # ── Coba select bulan & tahun terpisah ──
    selects = driver.find_elements(By.TAG_NAME, "select")
    for sel_el in selects:
        attr = (
            (sel_el.get_attribute("name") or "") + " " +
            (sel_el.get_attribute("id")   or "")
        ).lower()
        try:
            s = Select(sel_el)
            if "bulan" in attr or "month" in attr:
                # Coba pilih berdasarkan nilai angka bulan
                s.select_by_value(str(tgl.month))
            elif "tahun" in attr or "year" in attr:
                s.select_by_value(str(tgl.year))
            elif "tanggal" in attr or "tgl" in attr or "day" in attr:
                s.select_by_value(str(tgl.day))
        except Exception:
            pass


def _klik_cari(driver):
    from selenium.webdriver.common.by import By
    kw = ["cari", "search", "filter", "tampil", "submit", "lihat", "terapkan"]
    for el in (
        driver.find_elements(By.TAG_NAME, "button") +
        driver.find_elements(By.CSS_SELECTOR, "input[type='submit']") +
        driver.find_elements(By.CSS_SELECTOR, "input[type='button']")
    ):
        teks = (el.text or el.get_attribute("value") or "").lower()
        if any(k in teks for k in kw):
            try:
                el.click()
                return
            except Exception:
                pass


# =============================================================================
# PROGRESS
# =============================================================================

def baca_progress() -> date | None:
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as f:
            teks = f.read().strip()
        try:
            return date.fromisoformat(teks) + timedelta(days=1)
        except ValueError:
            pass
    return None


def tulis_progress(tanggal_str: str):
    with open(PROGRESS_FILE, "w") as f:
        f.write(tanggal_str)


# =============================================================================
# MAIN
# =============================================================================

def main():
    tgl_mulai = date(TAHUN_MULAI, 1, 1)
    tgl_akhir = date(TAHUN_AKHIR, 12, 31)

    # Lanjut dari progress jika ada
    lanjut = baca_progress()
    if lanjut and lanjut > tgl_mulai:
        print(f"[INFO] Melanjutkan dari {lanjut} (progress.txt ditemukan)")
        tgl_mulai = lanjut

    semua_tgl = []
    t = tgl_mulai
    while t <= tgl_akhir:
        semua_tgl.append(t)
        t += timedelta(days=1)

    total = len(semua_tgl)
    print(f"\n🛒  Scraping Bahan Pokok MAKANAN — Kab. Jember")
    print(f"    Periode : {tgl_mulai} s/d {tgl_akhir}")
    print(f"    Total   : {total} hari")
    print(f"    Output  : {OUTPUT_FILE}\n")

    driver = init_driver()

    from selenium.webdriver.support.ui import WebDriverWait
    wait = WebDriverWait(driver, 15)

    # Setup filter awal (pilih Jember)
    setup_filter(driver, wait)

    # Buka/append CSV
    file_baru = not os.path.exists(OUTPUT_FILE)
    csvfile = open(OUTPUT_FILE, "a", newline="", encoding="utf-8-sig")
    writer  = csv.DictWriter(csvfile, fieldnames=KOLOM)
    if file_baru:
        writer.writeheader()

    try:
        for i, tgl in enumerate(semua_tgl, 1):
            tgl_str = tgl.isoformat()
            pct     = i / total * 100
            print(f"[{i:4d}/{total}] {tgl_str} ({pct:5.1f}%) ... ", end="", flush=True)

            try:
                data = fetch_tanggal(driver, wait, tgl_str)
                if data:
                    writer.writerows(data)
                    csvfile.flush()
                    print(f"{len(data)} baris")
                else:
                    print("(kosong)")
            except Exception as e:
                print(f"ERROR: {e}")
                # Coba refresh driver jika crash
                try:
                    driver.quit()
                except Exception:
                    pass
                time.sleep(5)
                driver = init_driver()
                wait   = WebDriverWait(driver, 15)
                setup_filter(driver, wait)

            tulis_progress(tgl_str)
            time.sleep(DELAY_DETIK)

    except KeyboardInterrupt:
        print("\n\n[INFO] Dihentikan. Jalankan lagi untuk melanjutkan (progress.txt tersimpan).")

    finally:
        csvfile.close()
        try:
            driver.quit()
        except Exception:
            pass

    print(f"\n✅ Selesai! → {OUTPUT_FILE}")

    # Preview
    print("\n--- PREVIEW (10 baris pertama) ---")
    with open(OUTPUT_FILE, encoding="utf-8-sig") as f:
        for j, line in enumerate(f):
            print(line.rstrip())
            if j >= 10:
                break


if __name__ == "__main__":
    main()